# 03 — Forward-Return Targets

This notebook defines the future outcomes to predict and joins them to the existing raw-feature export. It provides theory and empty code cells for your implementation. Target construction intentionally uses future prices; those values must remain separate from the model inputs.

## 1. Load the price history and feature export

Use EDA_DATA_PATH for the complete price history and FTE_DATA_PATH for the eligible feature table. Parse date columns as dates and sort the price history by symbol and date. Preserve both tables separately.

Calculate future prices on the full price history before joining to eligible rows. Shifting an already filtered feature table can accidentally skip excluded dates and change a five-trading-observation horizon. Confirm unique (date, symbol) keys and consistent daily coverage.

In [1]:
import pandas as pd

from factor_model.config import EDA_DATA_PATH, FTE_DATA_PATH

eda_df = pd.read_csv(EDA_DATA_PATH, parse_dates=["date"])
fte_df = pd.read_csv(FTE_DATA_PATH, parse_dates=["date"])
display(eda_df)
display(fte_df)
eda_df.info()


,date,symbol,open,high,low,close,adjusted_close,volume,normalized_price,dollar_volume,daily_return,five_day_return,twenty_day_return,daily_rank,five_day_rank,twenty_day_rank
0,2018-01-02,AAPL,42.540001,43.075001,42.314999,43.064999,40.232380,102223600,100.000000,4.402259e+09,NaN,NaN,NaN,NaN,NaN,NaN
1,2018-01-03,AAPL,43.132500,43.637501,42.990002,43.057499,40.225365,118071600,99.982563,5.083868e+09,-0.000174,NaN,NaN,9.0,NaN,NaN
2,2018-01-04,AAPL,43.134998,43.367500,43.020000,43.257500,40.412201,89738400,100.446956,3.881859e+09,0.004645,NaN,NaN,5.0,NaN,NaN
3,2018-01-05,AAPL,43.360001,43.842499,43.262501,43.750000,40.872326,94640000,101.590624,4.140500e+09,0.011386,NaN,NaN,3.0,NaN,NaN
4,2018-01-08,AAPL,43.587502,43.902500,43.482498,43.587502,40.720512,82271200,101.213283,3.585996e+09,-0.003714,NaN,NaN,10.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17605,2024-12-24,XOM,106.519997,107.190002,105.699997,106.400002,100.614822,7807000,174.040549,8.306648e+08,0.000941,-0.014906,-0.113112,10.0,6.0,10.0
17606,2024-12-26,XOM,106.519997,107.029999,105.940002,106.489998,100.699928,9652400,174.187763,1.027884e+09,0.000846,0.000658,-0.097313,4.0,9.0,9.0
17607,2024-12-27,XOM,106.300003,107.989998,105.769997,106.480003,100.690483,11943900,174.171425,1.271787e+09,-0.000094,0.009193,-0.095020,1.0,7.0,9.0
17608,2024-12-30,XOM,106.300003,106.559998,105.510002,105.760002,100.009628,11080800,172.993702,1.171905e+09,-0.006762,-0.001039,-0.103425,3.0,3.0,9.0


,date,symbol,open,high,low,close,adjusted_close,volume,normalized_price,dollar_volume,daily_return,five_day_return,twenty_day_return,daily_rank,five_day_rank,twenty_day_rank,volatility_20,relative_volume_20,average_dollar_volume_20
0,2018-01-31,AAPL,41.717499,42.110001,41.625000,41.857498,39.104294,129915600,97.196074,5.437942e+09,0.002755,-0.038974,-0.028039,5.0,10.0,10.0,0.009462,1.035678,5.504557e+09
1,2018-02-01,AAPL,41.792500,42.154999,41.689999,41.945000,39.186062,188923200,97.399314,7.924384e+09,0.002091,-0.019461,-0.025837,4.0,6.0,10.0,0.009490,1.489640,5.646583e+09
2,2018-02-02,AAPL,41.500000,41.700001,40.025002,40.125000,37.485771,346375200,93.173139,1.389830e+10,-0.043390,-0.064194,-0.072415,9.0,10.0,10.0,0.013250,2.656918,6.147405e+09
3,2018-02-05,AAPL,39.775002,40.970001,39.000000,39.122501,36.549206,290954000,90.845249,1.138285e+10,-0.024985,-0.068290,-0.105771,2.0,4.0,10.0,0.013567,2.031814,6.509523e+09
4,2018-02-06,AAPL,38.707500,40.930000,38.500000,40.757500,38.076660,272975200,94.641829,1.112579e+10,0.041792,-0.023597,-0.064927,2.0,2.0,8.0,0.017207,1.783979,6.886512e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17405,2024-12-24,XOM,106.519997,107.190002,105.699997,106.400002,100.614822,7807000,174.040549,8.306648e+08,0.000941,-0.014906,-0.113112,10.0,6.0,10.0,0.008893,0.428766,1.921223e+09
17406,2024-12-26,XOM,106.519997,107.029999,105.940002,106.489998,100.699928,9652400,174.187763,1.027884e+09,0.000846,0.000658,-0.097313,4.0,9.0,9.0,0.008640,0.558930,1.885158e+09
17407,2024-12-27,XOM,106.300003,107.989998,105.769997,106.480003,100.690483,11943900,174.171425,1.271787e+09,-0.000094,0.009193,-0.095020,1.0,7.0,9.0,0.008696,0.702142,1.883569e+09
17408,2024-12-30,XOM,106.300003,106.559998,105.510002,105.760002,100.009628,11080800,172.993702,1.171905e+09,-0.006762,-0.001039,-0.103425,3.0,3.0,9.0,0.008521,0.649751,1.886567e+09


<class 'pandas.DataFrame'>
RangeIndex: 17610 entries, 0 to 17609
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   date               17610 non-null  datetime64[us]
 1   symbol             17610 non-null  str           
 2   open               17610 non-null  float64       
 3   high               17610 non-null  float64       
 4   low                17610 non-null  float64       
 5   close              17610 non-null  float64       
 6   adjusted_close     17610 non-null  float64       
 7   volume             17610 non-null  int64         
 8   normalized_price   17610 non-null  float64       
 9   dollar_volume      17610 non-null  float64       
 10  daily_return       17600 non-null  float64       
 11  five_day_return    17560 non-null  float64       
 12  twenty_day_return  17410 non-null  float64       
 13  daily_rank         17600 non-null  float64       
 14  five_day_rank    

## 2. Define the prediction target

Suppose today's adjusted close is 100 and the adjusted close five trading observations later is 105. The 5-day forward return is 5%. This is the outcome your model will eventually try to rank across stocks.

Use an initial horizon of five trading observations and the definition forward_return_5(t) = adjusted_close(t+5) / adjusted_close(t) − 1. Past five_day_return is an input; forward_return_5 is a target and must never appear in the feature list. With complete daily coverage, observations correspond to trading days rather than calendar days.

In [2]:
target_horizon = 5

## 3. Locate the future price and label end date

For each symbol, align its adjusted close five observations ahead with the current row. A negative shift moves future observations onto earlier row labels; perform the shift inside each symbol group so stocks never borrow one another's prices.

Also retain label_end_date, the actual future date used by the target. This makes the horizon inspectable and later allows training labels that extend into validation or test periods to be excluded. The final five rows of each stock's full history naturally lack future prices and label end dates.

In [3]:
prices = eda_df.sort_values(["symbol", "date"])[["symbol", "date", "adjusted_close"]]

future_price = (
    prices.groupby("symbol")["adjusted_close"].shift(-target_horizon)
) 
prices["label_end_date"] = (
  prices.groupby("symbol")["date"].shift(-target_horizon)
)

## 4. Calculate and verify forward returns

Calculate forward_return_5 using the aligned future adjusted close and current adjusted close. A rise from 100 to 105 should produce 0.05; a fall to 95 should produce −0.05.

Verify one result manually from two prices five observations apart. Check the corresponding label end date, expected missing values at each stock's history end, and absence of infinite returns. Missing future outcomes cannot be filled with zero: zero would mean an observed flat return.

In [4]:
prices["forward_return_5d"] = future_price / prices["adjusted_close"] - 1

## 5. Join outcomes to eligible feature rows\n",    
- Join the target fields onto the feature export using both date and symbol, with a one-to-one join validation. 
- Preserve the feature rows first so missing targets are visible, then exclude observations whose future return is unavailable from the labeled research dataset.,
- Check row counts before and after joining, unmatched keys, and unique keys. 
- Inspect eligible symbol counts on each retained date. 
- For the current fixed-universe experiment, use dates with a valid outcome for all ten feature-eligible stocks before calculating target ranks. 
- Future-return magnitudes must not determine feature eligibility.

In [5]:
labeled_df = fte_df.merge(
    prices[["date", "symbol", "forward_return_5d","label_end_date"]],
    on=["date", "symbol"],
    how="left",
    validate="one_to_one"
)
labeled_df = labeled_df.dropna(
  subset=["forward_return_5d", "label_end_date"]
).copy()
labeled_df.info()

<class 'pandas.DataFrame'>
Index: 17360 entries, 0 to 17404
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   date                      17360 non-null  datetime64[us]
 1   symbol                    17360 non-null  str           
 2   open                      17360 non-null  float64       
 3   high                      17360 non-null  float64       
 4   low                       17360 non-null  float64       
 5   close                     17360 non-null  float64       
 6   adjusted_close            17360 non-null  float64       
 7   volume                    17360 non-null  int64         
 8   normalized_price          17360 non-null  float64       
 9   dollar_volume             17360 non-null  float64       
 10  daily_return              17360 non-null  float64       
 11  five_day_return           17360 non-null  float64       
 12  twenty_day_return         17360 no

## 6. Rank future returns within each date

If AAPL's next-five-day return is 5%, MSFT's is 2%, and JPM's is −1%, AAPL has the largest observed future return. Convert forward returns into a target percentile within each date, with larger returns receiving larger percentiles and average ranks resolving ties.

Name the column target_percentile_5. With ten distinct values, percentiles range from 0.1 to 1.0. Higher means better future relative performance. Keep forward_return_5 for interpretation and later evaluation. These ranks are training outcomes, not scores a model could know at prediction time.

In [6]:
labeled_df["target_percentile_5d"] = (
  labeled_df.groupby("date")["forward_return_5d"]
    .rank(method="average", ascending=True, pct=True)
)

## 7. Inspect targets and understand overlapping horizons

Inspect a few complete date-level comparisons showing symbol, forward return, target percentile, and label end date. Review the largest gains and losses and verify that percentile ordering agrees with the underlying returns. Confirm that every label end date is later than its feature date.

Daily five-day targets overlap: Monday's and Tuesday's outcomes share several return intervals. They are therefore not independent observations. Later validation must use chronological boundaries and remove training rows whose label end dates reach the next evaluation period. Avoid random row splits and do not treat this notebook's full-history target summaries as out-of-sample model evidence.

The close-to-close target is a research outcome. An after-close prediction cannot assume a trade filled at that same closing price; executable entry timing and costs belong in the later backtest.

In [7]:
assert (
  labeled_df["label_end_date"] > labeled_df["date"]
).all(), "Date of target has to be after the start date"

selected_date = labeled_df["date"].min()

display(
  labeled_df.loc[
    labeled_df["date"] == selected_date,
    ["symbol", "forward_return_5d", "target_percentile_5d", "label_end_date"],
  ].sort_values("forward_return_5d")
)

,symbol,forward_return_5d,target_percentile_5d,label_end_date
15669,XOM,-0.118671,0.1,2018-02-07
12187,NEE,-0.065206,0.2,2018-02-07
10446,MSFT,-0.056836,0.3,2018-02-07
3482,CAT,-0.051849,0.4,2018-02-07
6964,JNJ,-0.048991,0.5,2018-02-07
5223,HD,-0.047835,0.6,2018-02-07
0,AAPL,-0.047124,0.7,2018-02-07
1741,AMGN,-0.046117,0.8,2018-02-07
13928,WMT,-0.035178,0.9,2018-02-07
8705,JPM,-0.024207,1.0,2018-02-07


## 8. Save and document the labeled dataset

Save the joined feature and target table to a new processed-data file, leaving both EDA and feature exports available. Add the new output path to config.py when implementing this section.

Keep an explicit list of model-input columns separate from forward_return_5, target_percentile_5, future adjusted close, and label_end_date. No future-derived column belongs in the model inputs.

Record the horizon, adjusted-price convention, ranking direction, missing-target treatment, retained date range, universe coverage, and successful checks. The next notebook will define chronological training, validation, and test periods before fitting a simple baseline.

In [8]:
from factor_model.config import LABELED_DATA_PATH

assert not labeled_df.duplicated(["date", "symbol"]).any()
assert labeled_df[
  ["forward_return_5d", "target_percentile_5d", "label_end_date"]
].notna().all().all()

labeled_df.to_csv(LABELED_DATA_PATH, index=False)

## Target construction findings

Record your target definition, retained observations, verification results, and exported file here.